In [237]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# Data import and analysis

In [238]:
df = pd.read_csv('../dataset/dirty_cafe_sales.csv')

In [239]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [240]:
# Info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [241]:
# Describe
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [242]:
# Missing values
df.isna().sum()


Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

# Cleaning data

## Item

In [243]:
# Item column clenup
cleanup_df = df.copy()

invalid_items = ['UNKNOWN', 'ERROR']

mask = (
        cleanup_df['Item'].isin(invalid_items)
        | cleanup_df['Item'].isna()
)

price_column = pd.to_numeric(cleanup_df['Price Per Unit'], errors='coerce')

rows_to_drop = []

for i in cleanup_df.loc[mask].index:
    row = cleanup_df.loc[i]

    price = pd.to_numeric(row['Price Per Unit'], errors='coerce')

    if pd.isna(price):
        total = pd.to_numeric(row['Total Spent'], errors='coerce')
        quantity = pd.to_numeric(row['Quantity'], errors='coerce')

        if pd.notna(total) and pd.notna(quantity) and quantity != 0:
            price = total / quantity
        else:
            rows_to_drop.append(i)
            continue

    match = cleanup_df[
        (abs(price_column - price) < 0.01)
        & cleanup_df['Item'].notna()
        & (~cleanup_df['Item'].isin(invalid_items))
        & (cleanup_df.index != i)
        ]

    if not match.empty:
        cleanup_df.loc[i, 'Item'] = match.iloc[0]['Item']

cleanup_df = cleanup_df.drop(index=rows_to_drop).reset_index(drop=True)

df = cleanup_df.copy()

In [244]:
# After Item column cleanup check. 2 rows were dropped as there was no way to determine the value that was supposed to be there
df.isna().sum()

Transaction ID         0
Item                   0
Quantity             138
Price Per Unit       177
Total Spent          172
Payment Method      2577
Location            3262
Transaction Date     159
dtype: int64

## Price Per Unit

In [245]:
# Price Per Unit column cleanup
price_cleanup_df = df.copy()
price_cleanup_df['Price Per Unit'] = pd.to_numeric(
    price_cleanup_df['Price Per Unit'],
    errors='coerce'
)

rows_to_drop = []

mask = price_cleanup_df['Price Per Unit'].isna()

for i in price_cleanup_df.loc[mask].index:
    row = price_cleanup_df.loc[i]

    total = pd.to_numeric(row['Total Spent'], errors='coerce')
    quantity = pd.to_numeric(row['Quantity'], errors='coerce')

    if pd.notna(total) and pd.notna(quantity) and quantity != 0:
        price_cleanup_df.loc[i, 'Price Per Unit'] = total / quantity
        continue

    match = price_cleanup_df[
        (price_cleanup_df['Item'] == row['Item'])
        & (price_cleanup_df['Price Per Unit'].notna())
        & (price_cleanup_df.index != i)
    ]

    if not match.empty:
        price_cleanup_df.loc[i, 'Price Per Unit'] = match.iloc[0]['Price Per Unit']
        continue

    rows_to_drop.append(i)

df = price_cleanup_df.copy()


In [246]:
# no rows have been dropped
df.isna().sum()

Transaction ID         0
Item                   0
Quantity             138
Price Per Unit         0
Total Spent          172
Payment Method      2577
Location            3262
Transaction Date     159
dtype: int64

## Quantity

In [247]:
# Quantity column cleanup
quantity_cleanup_df = df.copy()
quantity_cleanup_df['Quantity'] = pd.to_numeric(
    quantity_cleanup_df['Quantity'],
    errors='coerce'
)

rows_to_drop = []


mask = (quantity_cleanup_df['Quantity'].isna())

for i in quantity_cleanup_df.loc[mask].index:
    row = quantity_cleanup_df.loc[i]

    quantity = quantity_cleanup_df['Quantity'].iloc[i]

    if pd.isna(quantity):
        price = pd.to_numeric(row['Price Per Unit'], errors='coerce')
        total = pd.to_numeric(row['Total Spent'], errors='coerce')

        if pd.notna(price) and pd.notna(total) and price != 0:
            quantity = total / price
            quantity_cleanup_df.loc[i, 'Quantity'] = quantity
            continue

    match = quantity_cleanup_df[
        (quantity_cleanup_df['Item'] == row['Item'])
        & (quantity_cleanup_df['Total Spent'] == row['Total Spent'])
        & (quantity_cleanup_df['Quantity'].notna())
        & (quantity_cleanup_df.index != i)
    ]

    if not match.empty:
        quantity_cleanup_df.loc[i, 'Quantity'] = match.iloc[0]['Quantity']
        continue

    rows_to_drop.append(i)

quantity_cleanup_df = quantity_cleanup_df.drop(index=rows_to_drop).reset_index(drop=True)

df = quantity_cleanup_df.copy()


In [248]:
# 9 rows were dropped as there was no way to determine the value that was supposed to be there
df.isna().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent          167
Payment Method      2575
Location            3262
Transaction Date     159
dtype: int64

## Total Spent

In [249]:
# Total Spent cleanup
total_cleanup_df = df.copy()
total_cleanup_df['Price Per Unit'] = pd.to_numeric(
    total_cleanup_df['Price Per Unit'],
    errors='coerce'
)
total_cleanup_df['Quantity'] = pd.to_numeric(
    total_cleanup_df['Quantity'],
    errors='coerce'
)
total_cleanup_df['Total Spent'] = pd.to_numeric(
    total_cleanup_df['Total Spent'],
    errors='coerce'
)

mask = total_cleanup_df['Total Spent'].isna()

total_cleanup_df.loc[mask, 'Total Spent'] = (
    total_cleanup_df.loc[mask, 'Price Per Unit']
    * total_cleanup_df.loc[mask, 'Quantity']
)

df = total_cleanup_df.copy()


## Payment Method

In [250]:
cols = ['Price Per Unit', 'Total Spent', 'Quantity', 'Item', 'Payment Method', 'Location', 'Transaction Date']
temp = df[cols].copy()

for c in ['Price Per Unit', 'Total Spent', 'Quantity']:
    temp[c] = (
        temp[c]
        .astype(str)
        .str.replace(r'[^0-9.\-]', '', regex=True)
        .replace({'': np.nan})
    )
    temp[c] = pd.to_numeric(temp[c], errors='coerce')

temp_ts = pd.to_datetime(temp['Transaction Date'], errors='coerce')

## Location

In [251]:
cols = ['Price Per Unit', 'Total Spent', 'Quantity', 'Item', 'Payment Method', 'Location', 'Transaction Date']
temp = df[cols].copy()

for c in ['Price Per Unit', 'Total Spent', 'Quantity']:
    temp[c] = (
        temp[c]
        .astype(str)
        .str.replace(r'[^0-9.\-]', '', regex=True)
        .replace({'': np.nan})
    )
    temp[c] = pd.to_numeric(temp[c], errors='coerce')

temp_ts = pd.to_datetime(temp['Transaction Date'], errors='coerce')

## Transaction Date

In [252]:
cols = ['Price Per Unit', 'Total Spent', 'Quantity', 'Item', 'Payment Method', 'Location', 'Transaction Date']
temp = df[cols].copy()

for c in ['Price Per Unit', 'Total Spent', 'Quantity']:
    temp[c] = (
        temp[c]
        .astype(str)
        .str.replace(r'[^0-9.\-]', '', regex=True)
        .replace({'': np.nan})
    )
    temp[c] = pd.to_numeric(temp[c], errors='coerce')

temp_ts = pd.to_datetime(temp['Transaction Date'], errors='coerce')


## General cleaning: duplicates and zero-only columns

In [253]:
gen_df = df.copy()

helpers = [c for c in gen_df.columns if c.endswith(('_enc', '_imputed', '_parsed', '_ts')) or c in ('Transaction Date_imputed', 'Transaction_ts_imputed')]
if helpers:
    gen_df = gen_df.drop(columns=helpers)

gen_df = gen_df.drop_duplicates().reset_index(drop=True)

numeric_cols = gen_df.select_dtypes(include=[np.number]).columns.tolist()

cols_to_drop = [c for c in gen_df.columns if gen_df[c].nunique(dropna=True) <= 1]
cols_to_drop += [c for c in numeric_cols if gen_df[c].fillna(0).eq(0).all() and c not in cols_to_drop]
cols_to_drop = list(dict.fromkeys(cols_to_drop))
if cols_to_drop:
    gen_df = gen_df.drop(columns=cols_to_drop)

is_all_unknown = gen_df.replace('unknown', np.nan).isna().all(axis=1)

is_all_zero_numeric = False
if numeric_cols:
    is_all_zero_numeric = (gen_df[numeric_cols].fillna(0).sum(axis=1) == 0)

drop_mask = is_all_unknown | is_all_zero_numeric
if drop_mask.any():
    gen_df = gen_df.loc[~drop_mask].reset_index(drop=True)

final_df = gen_df.copy()
for col in ['Payment Method', 'Location', 'Transaction Date']:
    if col in final_df.columns:
        final_df[col] = final_df[col].replace({None: np.nan})
        final_df[col] = final_df[col].fillna('unknown')

if 'Transaction Date' in final_df.columns:
    parsed = pd.to_datetime(final_df['Transaction Date'], errors='coerce')
    final_df['Transaction Date'] = parsed.dt.strftime('%Y-%m-%d %H:%M:%S')
    final_df['Transaction Date'] = final_df['Transaction Date'].fillna('unknown')

final_df.to_csv('../dataset/dirty_cafe_sales_cleaned.csv', index=False)


In [255]:
cleaned_df = pd.read_csv('../dataset/dirty_cafe_sales_cleaned.csv')
cleaned_df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64